# 🖼️ DDIM Latent Space Image Interpolation
Interpolate between two images (e.g., man → woman) using DDIM inversion + latent interpolation.

**How it works:**
1. Encode both images into the diffusion model's latent space via DDIM inversion
2. Spherically interpolate (slerp) between the two latent codes
3. Decode each interpolated latent back to an image

This gives smooth, semantically meaningful transitions — exactly like in the DDIM paper.

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q diffusers transformers accelerate Pillow

In [ ]:
# ── Step 2: Imports ───────────────────────────────────────────────────────────
import torch
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# ── Step 3: Load Stable Diffusion with DDIM scheduler ─────────────────────────
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
)
pipe.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # saves VRAM

vae       = pipe.vae
unet      = pipe.unet
tokenizer = pipe.tokenizer
text_enc  = pipe.text_encoder
scheduler = pipe.scheduler
print("Model loaded ✓")

In [ ]:
# ── Step 4: Helper functions ──────────────────────────────────────────────────

def load_image(path_or_url, size=(512, 512)):
    """Load from local path or URL, resize, return PIL Image."""
    if path_or_url.startswith("http"):
        response = requests.get(path_or_url)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(path_or_url).convert("RGB")
    return img.resize(size)


def encode_image(pil_img):
    """Encode a PIL image to VAE latent space."""
    dtype = torch.float16 if device == "cuda" else torch.float32
    x = torch.tensor(np.array(pil_img)).permute(2, 0, 1).unsqueeze(0)
    x = (x.float() / 127.5 - 1.0).to(device, dtype=dtype)
    with torch.no_grad():
        latent = vae.encode(x).latent_dist.mean * 0.18215
    return latent


def decode_latent(latent):
    """Decode a VAE latent back to a PIL image."""
    with torch.no_grad():
        img = vae.decode(latent / 0.18215).sample
    img = (img.clamp(-1, 1) + 1) / 2
    img = img.squeeze(0).permute(1, 2, 0).cpu().float().numpy()
    return Image.fromarray((img * 255).astype(np.uint8))


@torch.no_grad()
def get_text_embedding(prompt):
    """Get CLIP text embedding for a prompt (used as conditioning)."""
    dtype = torch.float16 if device == "cuda" else torch.float32
    tokens = tokenizer(prompt, return_tensors="pt",
                       padding="max_length", max_length=77,
                       truncation=True).input_ids.to(device)
    return text_enc(tokens)[0].to(dtype)


@torch.no_grad()
def ddim_inversion(latent, prompt_emb, num_steps=50):
    """Invert a latent to noise space using DDIM forward process."""
    scheduler.set_timesteps(num_steps)
    latent = latent.clone()
    # Walk forward in time (inversion = noise adding)
    for t in reversed(scheduler.timesteps):
        noise_pred = unet(latent, t, encoder_hidden_states=prompt_emb).sample
        latent = scheduler.step(noise_pred, t, latent).prev_sample
    return latent


@torch.no_grad()
def ddim_sample(noisy_latent, prompt_emb, num_steps=50):
    """Denoise a latent using DDIM reverse process."""
    scheduler.set_timesteps(num_steps)
    latent = noisy_latent.clone()
    for t in scheduler.timesteps:
        noise_pred = unet(latent, t, encoder_hidden_states=prompt_emb).sample
        latent = scheduler.step(noise_pred, t, latent).prev_sample
    return latent


def slerp(t, v0, v1):
    """Spherical linear interpolation between two latent tensors."""
    v0_f = v0.float().flatten()
    v1_f = v1.float().flatten()
    dot = torch.dot(v0_f / v0_f.norm(), v1_f / v1_f.norm()).clamp(-1, 1)
    theta = torch.acos(dot)
    if theta.abs() < 1e-6:
        return (1 - t) * v0 + t * v1
    return (torch.sin((1 - t) * theta) / torch.sin(theta)) * v0 + \
           (torch.sin(t * theta)       / torch.sin(theta)) * v1

print("Helpers defined ✓")

In [ ]:
# ── Step 5: Load your two images ──────────────────────────────────────────────
# 👉 Replace these with your own image paths or URLs!

# Option A – URLs (used by default)
URL_MAN   = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg"  # placeholder
URL_WOMAN = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"              # placeholder

# Option B – local upload paths (uncomment and set paths if you upload to Colab)
# from google.colab import files
# uploaded = files.upload()   # upload two files
# path_list = list(uploaded.keys())
# URL_MAN, URL_WOMAN = path_list[0], path_list[1]

img_a = load_image(URL_MAN)
img_b = load_image(URL_WOMAN)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_a); axes[0].set_title("Image A"); axes[0].axis("off")
axes[1].imshow(img_b); axes[1].set_title("Image B"); axes[1].axis("off")
plt.suptitle("Source Images"); plt.tight_layout(); plt.show()

In [ ]:
# ── Step 6: Encode images + DDIM inversion ────────────────────────────────────
NUM_STEPS = 50   # DDIM steps (higher = better quality, slower)

prompt_a = "a photo of a man, high quality, realistic"
prompt_b = "a photo of a woman, high quality, realistic"

emb_a = get_text_embedding(prompt_a)
emb_b = get_text_embedding(prompt_b)

print("Encoding images to VAE latents...")
lat_a = encode_image(img_a)
lat_b = encode_image(img_b)

print("Running DDIM inversion (image A)...")
noise_a = ddim_inversion(lat_a, emb_a, num_steps=NUM_STEPS)

print("Running DDIM inversion (image B)...")
noise_b = ddim_inversion(lat_b, emb_b, num_steps=NUM_STEPS)

print("Inversion done ✓")

In [ ]:
# ── Step 7: Interpolate and decode ────────────────────────────────────────────
NUM_FRAMES = 7   # how many steps in the interpolation (including endpoints)

alphas = np.linspace(0, 1, NUM_FRAMES)
frames = []

for i, alpha in enumerate(alphas):
    print(f"  Decoding frame {i+1}/{NUM_FRAMES}  (α={alpha:.2f})...")

    # Interpolate in noise space
    interp_noise = slerp(alpha, noise_a, noise_b)

    # Interpolate the text conditioning too (optional but helps)
    interp_emb = slerp(alpha, emb_a, emb_b)

    # Decode back to image via DDIM sampling
    interp_lat = ddim_sample(interp_noise, interp_emb, num_steps=NUM_STEPS)
    frames.append(decode_latent(interp_lat))

print("All frames generated ✓")

In [ ]:
# ── Step 8: Display the interpolation strip ───────────────────────────────────
fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(3 * NUM_FRAMES, 3))
for ax, frame, alpha in zip(axes, frames, alphas):
    ax.imshow(frame)
    ax.set_title(f"α={alpha:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("DDIM Latent Space Interpolation (A → B)", fontsize=13)
plt.tight_layout()
plt.savefig("interpolation_strip.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to interpolation_strip.png")

In [ ]:
# ── Step 9 (bonus): Save as animated GIF ─────────────────────────────────────
gif_frames = frames + frames[-2:0:-1]   # ping-pong loop
gif_frames[0].save(
    "interpolation.gif",
    save_all=True,
    append_images=gif_frames[1:],
    duration=200,
    loop=0,
)
print("GIF saved to interpolation.gif")

# Display inline
from IPython.display import Image as IPImage
IPImage("interpolation.gif")